In [ ]:
#!/usr/bin/env python
# coding: utf-8

# ----------------------------------------
# CELDA 1: PREPARACIÓN DE DATOS (Corregida)
# ----------------------------------------

"""Procesamiento de datos con segmentacion y ruido.

Esta funcion lee los archivos originales, los divide en segmentos,
introduce ruido gaussiano y devuelve un DataFrame con una unica
observacion por intervalo de tiempo.
"""

from __future__ import annotations

from pathlib import Path
import re
import io

import numpy as np
import pandas as pd
import math

# ──────────────────────────────
# Helpers robustos de lectura
# ──────────────────────────────
def _clean_numeric_lines(path: Path) -> str:
    """Devuelve solo filas NUMÉRICAS (3 columnas) del archivo:
       - ignora comentarios (#), vacías y cabeceras (e.g., 'time ...')
       - normaliza el “−” unicode a '-'."""
    out = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            s = s.replace("\u2212", "-")  # “−” → "-"
            parts = s.split()
            if len(parts) < 3:
                continue
            try:
                float(parts[0]); float(parts[1]); float(parts[2])
            except ValueError:
                continue
            out.append(" ".join(parts[:3]))
    return "\n".join(out)

def _load_three_cols(path: Path) -> np.ndarray:
    buf = _clean_numeric_lines(path)
    if not buf:
        raise ValueError(f"{path} no contiene filas numéricas válidas.")
    arr = np.loadtxt(io.StringIO(buf))
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    if arr.shape[1] != 3:
        raise ValueError(f"{path} tiene {arr.shape[1]} columnas; se esperaban 3.")
    return arr

# ──────────────────────────────
# Filtro por ventana de meses
# ──────────────────────────────
def filtrar_por_ventana_meses(
    df: pd.DataFrame,
    *,
    col_dias: str = 'time_days',
    mes_inicial: int = 1,
    n_meses: int = 6,
    dias_por_anio: int = 365,
    agregar_aux: bool = False
) -> pd.DataFrame:
    if col_dias not in df.columns:
        raise ValueError(f"La columna '{col_dias}' no existe en el DataFrame.")
    dias_mes = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    limites = np.concatenate(([0], np.cumsum(dias_mes)))
    dias_abs = df[col_dias].astype(int).to_numpy()
    anio_rel = dias_abs // dias_por_anio
    dia_anio = dias_abs % dias_por_anio
    mes = np.searchsorted(limites, dia_anio, side='right')
    meses_ok = [((mes_inicial - 1 + i) % 12) + 1 for i in range(n_meses)]
    keep = np.isin(mes, meses_ok)
    out = df.loc[keep].copy().reset_index(drop=True)
    if agregar_aux:
        out['anio_rel'] = anio_rel[keep]
        out['mes'] = mes[keep]
    return out

# ──────────────────────────────
# Preparación de datos ROBUSTA
# ──────────────────────────────
def preparar_datos(
    ruido: float,
    frecuencia_observacion: float,
    periodo_orbital: float,
    numero_segmentos: int,
    n_meses: int = 6,
) -> pd.DataFrame:
    """Devuelve un DataFrame con una observacion por intervalo."""

    raw_files = [
        "data/original/mdot_series_q010_e000.txt",
        "data/original/mdot_series_q025_e000.txt",
        "data/original/mdot_series_q050_e000.txt",
        "data/original/mdot_series_q100_e000.txt",
        "data/original/mdot_series_q100_e050.txt",
        "data/original/mdot_series_q100_e040.txt",
        "data/original/mdot_series_q100_e030.txt",
        "data/original/mdot_series_q100_e020.txt",
        "data/original/mdot_series_q100_e010.txt",
        "data/original/mdot_series_q100_e060.txt",
        "data/original/mdot_series_q100_e070.txt",
    ]

    segments_dir = Path("data/segments")
    segments_dir.mkdir(parents=True, exist_ok=True)

    # Limpia segmentos viejos (evita arrastrar cabeceras antiguas o residuos)
    for old in segments_dir.glob("*_seg??.txt"):
        try:
            old.unlink()
        except Exception:
            pass

    float_fmt = "%.18e"

    # 1) Troceo con NumPy (solo 3 columnas float)
    for raw in raw_files:
        raw_path = Path(raw)
        if not raw_path.exists():
            continue
        arr = _load_three_cols(raw_path)
        n = arr.shape[0]
        rows_per_seg = max(1, n // numero_segmentos)
        for i in range(numero_segmentos):
            start = i * rows_per_seg
            end = (i + 1) * rows_per_seg if i < numero_segmentos - 1 else n
            seg = arr[start:end]
            if seg.size == 0:
                continue
            seg_name = raw_path.stem + f"_seg{i+1:02d}" + raw_path.suffix
            seg_path = segments_dir / seg_name
            np.savetxt(seg_path, seg, fmt=float_fmt)

    # 2) Ensamblado de segmentos + cálculo
    pattern     = "*_seg??.txt"
    col_time    = "time"
    col_primary = "acre_rate_primary"
    col_second  = "acre_rate_secondary"
    col_mw      = "acre_rate_mass_weighted"

    def _parse(fname: str) -> tuple[float | None, float | None, int | None]:
        m = re.search(r"q(\d+)_e(\d+)_seg(\d+)", fname)
        if not m:
            return None, None, None
        q = int(m.group(1)) / 100.0
        e = int(m.group(2)) / 100.0
        s = int(m.group(3))
        return q, e, s

    frames: list[pd.DataFrame] = []
    for f in sorted(segments_dir.glob(pattern)):
        q_val, e_val, seg_id = _parse(f.name)
        if q_val is None:
            continue

        arr = _load_three_cols(f)
        df = pd.DataFrame(arr, columns=[col_time, col_primary, col_second])

        # Validación: time no debe ser constante
        if df[col_time].nunique(dropna=True) <= 1:
            raise ValueError(f"La columna '{col_time}' quedó constante en {f}.")

        # pesos por masa
        w1 = 1.0 / (1.0 + q_val)
        w2 = q_val / (1.0 + q_val)
        df[col_mw] = w1 * df[col_primary].abs() + w2 * df[col_second].abs()

        q_tag   = f"{int(round(q_val * 100)):03d}"
        e_tag   = f"{int(round(e_val * 100)):03d}"
        seg_tag = f"{seg_id:02d}"
        df["q"], df["e"], df["seg"], df["id"] = q_val, e_val, seg_id, f"{q_tag}_{e_tag}_{seg_tag}"

        frames.append(df)

    if not frames:
        raise RuntimeError("No se generaron segmentos válidos.")

    all_data = pd.concat(frames, ignore_index=True)
    all_data = all_data[~all_data["seg"].isin({1, 2})]

    # Ruido
    rng = np.random.default_rng()
    sigma = all_data[col_mw].mean()
    all_data[f"{col_mw}_clean"] = all_data[col_mw]
    noise = rng.normal(0.0, ruido * sigma, size=len(all_data))
    all_data[col_mw] = all_data[col_mw] + noise

    # Buckets y reducción a una observación por intervalo
    all_data["time_days"] = all_data[col_time] * periodo_orbital
    all_data["bucket"] = np.floor((all_data["time_days"] + 1e-9) / frecuencia_observacion).astype(int)
    unique_df = (
        all_data
        .groupby(["id", "bucket"], as_index=False)
        .median(numeric_only=True)
        .sort_values(["id", "time_days"])
        .reset_index(drop=True)
        .drop(columns="bucket")
    )

    unique_df = filtrar_por_ventana_meses(unique_df, mes_inicial=0, n_meses=n_meses)
    return unique_df


# --------------------------------------------
# CELDA 2: ANÁLISIS DE SIMILITUD (Modificada)
# --------------------------------------------

"""Analisis de similitud basado en el cuaderno analisis-RF."""
from joblib import Parallel, delayed
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from pycatch22 import catch22_all
import random  # <--- AÑADIDO

def _extract_catch22(id_: str, grp: pd.DataFrame) -> pd.Series:
    values = grp.sort_values("time")["acre_rate_mass_weighted"].to_numpy()
    res = catch22_all(values)
    return pd.Series(res["values"], index=res["names"], name=id_)

def _compute_features(df: pd.DataFrame, n_jobs: int = -1) -> pd.DataFrame:
    grouped = df.groupby("id", sort=False)
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_extract_catch22)(i, g) for i, g in grouped
    )
    features_df = pd.concat(results, axis=1).T
    features_df.reset_index(inplace=True)
    features_df.rename(columns={"index": "id"}, inplace=True)
    return features_df

def _standardize_features(df: pd.DataFrame) -> tuple[pd.DataFrame, StandardScaler]:
    feature_cols = df.columns.difference(["id"])
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df[feature_cols])
    df_std = pd.DataFrame(scaled, columns=feature_cols, index=df.index)
    df_std.insert(0, "id", df["id"])
    return df_std, scaler

def _weight_features(df_std: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    df_std[["q", "e", "seg"]] = df_std["id"].str.split("_", expand=True)
    df_std["q"] = df_std["q"].astype(int) / 100
    df_std["e"] = df_std["e"].astype(int) / 100
    df_std.drop(columns="seg", inplace=True)
    id_col = "id"
    target_cols = ["q", "e"]
    feature_cols = df_std.columns.difference([id_col] + target_cols)
    Y_enc = pd.DataFrame(index=df_std.index)
    for col in target_cols:
        le = LabelEncoder().fit(df_std[col])
        Y_enc[col] = le.transform(df_std[col])
    X = df_std[feature_cols]
    Y = Y_enc[target_cols]
    base = RandomForestClassifier(random_state=42, n_jobs=-1)
    multi = MultiOutputClassifier(base, n_jobs=-1)
    grid = GridSearchCV(
        multi,
        {
            "estimator__n_estimators": [200, 500],
            "estimator__max_depth": [None, 20, 40],
            "estimator__min_samples_leaf": [1, 2, 4],
            "estimator__max_features": ["sqrt", "log2"],
        },
        cv=5,
        scoring=None,
        n_jobs=-1,
    )
    best = grid.fit(X, Y).best_estimator_
    importances = np.vstack([est.feature_importances_ for est in best.estimators_])
    importances = importances.mean(axis=0)
    importances /= importances.sum()
    df_weighted = df_std.copy()
    df_weighted[feature_cols] = df_weighted[feature_cols].mul(importances, axis=1)
    return df_weighted, importances

def evaluar_similitud(
    df: pd.DataFrame,
    TARGET_GRUPO: str,
    n_samples: int,
    distancia: str = "manhattan",
) -> float:
    """Evalua la tasa de exito para n_samples aleatorios del TARGET_GRUPO."""
    features = _compute_features(df)
    df_std, _ = _standardize_features(features)
    df_w, _ = _weight_features(df_std)
    feature_cols = df_w.columns.difference(["id", "q", "e"])
    all_group_ids = df_w.loc[df_w["id"].str.startswith(TARGET_GRUPO + "_"), "id"].tolist()
    if not all_group_ids:
        return 0.0
    n_seleccionados = min(n_samples, len(all_group_ids))
    import random
    targets_seleccionados = random.sample(all_group_ids, n_seleccionados)
    exitos = 0
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler(feature_range=(0, 1))
    from scipy.spatial.distance import cdist
    for target_id in targets_seleccionados:
        x0 = df_w.loc[df_w["id"] == target_id, feature_cols].to_numpy()
        X_df = df_w.loc[df_w["id"] != target_id]
        X = X_df[feature_cols].to_numpy()
        other_ids = X_df["id"].to_numpy()
        metric = "cityblock" if distancia == "manhattan" else distancia
        kwargs = {}
        if distancia == "mahalanobis":
            if X.shape[0] > X.shape[1]:
                cov = np.cov(X, rowvar=False)
                kwargs["VI"] = np.linalg.inv(cov)
            else:
                metric = "cityblock"
        dist = cdist(X, x0, metric=metric, **kwargs).flatten()
        dist_df = pd.DataFrame({"id": other_ids, distancia: dist})
        dist_df[["q", "e", "seg"]] = dist_df["id"].str.split("_", expand=True)
        dist_df["q"] = dist_df["q"].astype(int) / 100
        dist_df["e"] = dist_df["e"].astype(int) / 100
        dist_df.drop(columns="seg", inplace=True)
        dist_df[[distancia]] = scaler.fit_transform(dist_df[[distancia]])
        median = (
            dist_df.groupby(["q", "e"], as_index=False)[distancia]
            .median()
            .sort_values(distancia)
        )
        median["id"] = median.apply(
            lambda r: f"{int(round(r['q'] * 100)):03d}_{int(round(r['e'] * 100)):03d}",
            axis=1,
        )
        if not median.empty and median.iloc[0]["id"] == TARGET_GRUPO:
            exitos += 1
    return exitos / n_seleccionados if n_seleccionados else 0.0

# --------------------------------------------
# CELDA 3: GRID SEARCH RUNNER (Modificada)
# --------------------------------------------
from itertools import product

def r(min_, max_, step):
    n = int(math.floor((max_ - min_) / step))
    values = [min_ + i * step for i in range(n + 1)]
    if values and values[-1] < max_:
        values.append(max_)
    return values

def correr_grid(
    preparar_datos_fn,
    evaluar_similitud_fn,
    ruido_rg, frec_rg, per_rg, nseg_rg, n_meses_rg,
    target_grupo,
    n_aleatorios,
    distancia="manhattan",
):
    filas = []
    fallidos = []
    total_combinaciones = (
        len(r(*ruido_rg)) * len(r(*frec_rg)) * len(r(*per_rg)) *
        len(r(*nseg_rg)) * len(r(*n_meses_rg))
    )
    i = 0
    for ruido, frec, per, nseg, n_mes in product(
        r(*ruido_rg), r(*frec_rg), r(*per_rg), r(*nseg_rg), r(*n_meses_rg)
    ):
        i += 1
        print(f"Estudiando combinacion {i}/{total_combinaciones}")
        try:
            df = preparar_datos_fn(ruido, frec, per, int(nseg), int(n_mes))
            ok = evaluar_similitud_fn(
                df,
                TARGET_GRUPO=target_grupo,
                n_samples=n_aleatorios,
                distancia=distancia
            )
            filas.append((ruido, frec, per, int(nseg), int(n_mes), ok))
        except ValueError as exc:
            if "NaN" in str(exc) or "Input contains NaN" in str(exc):
                filas.append((ruido, frec, per, int(nseg), int(n_mes), 0.0))
                fallidos.append((ruido, frec, per, int(nseg), int(n_mes)))
            else:
                raise
    return pd.DataFrame(
        filas,
        columns=["ruido", "frec_obs", "periodo", "n_seg", "n_meses", "ok"]
    ), fallidos

# --------------------------------------------
# CELDA 4: EJECUCIÓN Y GUARDADO (Modificada)
# --------------------------------------------
import time
_t0 = time.perf_counter()

ruido_rg = (0, 1.5, 0.1)
frec_rg  = (7, 28, 7)
per_rg   = (30, 360, 30)
nseg_rg  = (20, 20, 1)
nmes_rg  = (3, 12, 3)

TARGET_GRUPO = "100_050"
N_ALEATORIOS = 10

df_res_tupla = correr_grid(
    preparar_datos_fn   = preparar_datos,
    evaluar_similitud_fn= evaluar_similitud,
    ruido_rg = ruido_rg,
    frec_rg  = frec_rg,
    per_rg   = per_rg,
    nseg_rg  = nseg_rg,
    n_meses_rg = nmes_rg,
    target_grupo = TARGET_GRUPO,
    n_aleatorios = N_ALEATORIOS
)

print(f"⏱️  Notebook completo: {time.perf_counter()-_t0:,.2f} s")

df_only = df_res_tupla[0]
output_filename = f"data/sensibilidad/sensibilidad_GRUPO_{TARGET_GRUPO}.csv"
Path("data/sensibilidad").mkdir(parents=True, exist_ok=True)
df_only.to_csv(output_filename, index=False)
print(f"\nResultados guardados en: {output_filename}")

fallidos = df_res_tupla[1]
if fallidos:
    print(f"Se encontraron {len(fallidos)} combinaciones fallidas (registradas con ok=0.0):")
    for f in fallidos[:5]:
        print(f"  - {f}")


Estudiando combinacion 1/3072
Estudiando combinacion 2/3072
Estudiando combinacion 3/3072
Estudiando combinacion 4/3072
Estudiando combinacion 5/3072
Estudiando combinacion 6/3072
Estudiando combinacion 7/3072
Estudiando combinacion 8/3072
Estudiando combinacion 9/3072
Estudiando combinacion 10/3072
Estudiando combinacion 11/3072
Estudiando combinacion 12/3072
Estudiando combinacion 13/3072
Estudiando combinacion 14/3072
Estudiando combinacion 15/3072
Estudiando combinacion 16/3072
Estudiando combinacion 17/3072
Estudiando combinacion 18/3072
Estudiando combinacion 19/3072
Estudiando combinacion 20/3072
Estudiando combinacion 21/3072
Estudiando combinacion 22/3072
Estudiando combinacion 23/3072
Estudiando combinacion 24/3072
Estudiando combinacion 25/3072
Estudiando combinacion 26/3072
Estudiando combinacion 27/3072
Estudiando combinacion 28/3072
Estudiando combinacion 29/3072
Estudiando combinacion 30/3072
Estudiando combinacion 31/3072
Estudiando combinacion 32/3072
Estudiando combin